In [30]:
# imports and fixing version issues 
import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
!pip install -U sentence-transformers qdrant-client pandas pyarrow fastapi
!pip install "httpx<0.28" "pandas<3.0" --upgrade
!pip uninstall -y google-genai
!pip install "httpx<0.28" --upgrade


  Using cached pandas-3.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached pandas-3.0.0-cp313-cp313-macosx_11_0_arm64.whl (9.9 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lseg-data 2.1.1 requires pandas<3.0,>=2.0, but you have pandas 3.0.0 which is incompatible.
  Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (91 kB)
Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl (10.7 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.0
    Uninstalling pandas-3.0.0:
      Successfully uninstalled pandas-3.0.0


In [31]:
# Qdrant (Docker/Server mode) config 
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

COLLECTION_NAME = "banking77"
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"      

VECTOR_SIZE = 384
TOP_K = 3


In [32]:
# Initialise model used to create the embeddings, formatting the text pairs from the dataset as query:answer  
from sentence_transformers import SentenceTransformer

class EmbeddingModel:
    def __init__(self, model_name):
        self.model = SentenceTransformer(model_name)

    def encode(self, texts, is_query=False):
        if isinstance(texts, str):
            texts = [texts]

        prefix = "query: " if is_query else "passage: "
        texts = [prefix + t for t in texts]

        return self.model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True
        )


In [33]:
# importing the data and putting it in a df
import pandas as pd

url = "https://huggingface.co/datasets/PolyAI/banking77/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"
df = pd.read_parquet(url)

df.head(), len(df)


(                                                text  label
 0                     I am still waiting on my card?     11
 1  What can I do if my card still hasn't arrived ...     11
 2  I have been waiting over a week. Is the card s...     11
 3  Can I track my card while it is in the process...     11
 4  How do I know if I will get my card, or if it ...     11,
 10003)

In [34]:
# for troubleshooting the qdrant import
import os
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

print("Imported QdrantClient:", QdrantClient)


Imported QdrantClient: <class 'qdrant_client.qdrant_client.QdrantClient'>


In [35]:
# qdrant connection confirmation, so that each notebook can run indep and I can know if I ran the cell out of order
from qdrant_client import QdrantClient

# Create Qdrant client, instantce of the connection to qdrant (has not created a collection yet, just connecting)
client = QdrantClient(
    host=QDRANT_HOST,
    port=QDRANT_PORT
)

print(f" Connected to Qdrant at {QDRANT_HOST}:{QDRANT_PORT}")

 Connected to Qdrant at localhost:6333


In [36]:
# this can be run as is even if a model change is required, since the collection by the name of that dataset (if it exists) is being deleted and recreated
#from qdrant_client.models import VectorParams, Distance

# Delete old collection if it exists
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print("🗑️ Deleted existing collection")

# Create collection with correct vector size
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,   # 384
        distance=Distance.COSINE
    )
)

print(f" Collection '{COLLECTION_NAME}' created with VECTOR_SIZE = {VECTOR_SIZE}")

🗑️ Deleted existing collection
 Collection 'banking77' created with VECTOR_SIZE = 384


In [37]:
from qdrant_client.models import PointStruct
from tqdm import tqdm

embedder = EmbeddingModel(EMBEDDING_MODEL)

texts = df["text"].tolist()
labels = df["label"].tolist()

#generating embeddings for the passages 
embeddings = embedder.encode(texts)

BATCH_SIZE = 256  # safe default, can be changed, can try 512 also

for start in tqdm(range(0, len(texts), BATCH_SIZE)):
    end = start + BATCH_SIZE

# building a list of qdrant points
    batch_points = [
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload={
                "text": texts[i],
                "label": int(labels[i])
            }
        )
        for i in range(start, min(end, len(texts)))
    ]
# Inserts (or updates) points into Qdrant
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch_points
    )

print(" Data ingested into Qdrant (batched)")


100%|██████████| 40/40 [00:02<00:00, 13.78it/s]

 Data ingested into Qdrant (batched)


In [39]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

# semantic search fn
def semantic_search(query, top_k=TOP_K, label_filter=None):
    # Encode query
    query_vector = embedder.encode(query, is_query=True)[0]

    # Optional label filter, and class conditional semantic search
    q_filter = None
    if label_filter is not None:
        q_filter = Filter(
            must=[
                FieldCondition(
                    key="label",
                    match=MatchValue(value=int(label_filter))
                )
            ]
        )

    # Query Qdrant (vector search)
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=top_k,
        query_filter=q_filter,
        with_payload=True
    )

    # Server mode returns ScoredPoint list (returns semantic similarity score, payload, id)
    return list(results.points)


In [40]:
# def calculate_confidence(results):
#     if not results:
#         return 0.0

#     scores = [r.score for r in results]

#     if len(scores) == 1:
#         return round(scores[0], 3)

#     top, second = scores[0], scores[1]
#     relative_gap = max(top - second, 0.0)
#     normalized = top / max(sum(scores), 1e-6)

#     return round((0.7 * normalized) + (0.3 * relative_gap), 3)

def calculate_metrics(results):
#Separate similarity and confidence for clarity
    if not results:
        return {
            "top_similarity": 0.0,
            "confidence": 0.0,
            "status": "NO_RESULTS"
        }
    
    scores = [r.score for r in results]
    top_score = scores[0]
    
    # SIMILARITY: Just the raw cosine similarity
    similarity = round(top_score, 3)
    
    # CONFIDENCE: Your composite metric
    if len(scores) == 1:
        confidence = round(top_score, 3)
    else:
        second_score = scores[1]
        relative_gap = max(top_score - second_score, 0.0)
        normalized = top_score / max(sum(scores), 1e-6)
        confidence = round((0.7 * normalized) + (0.3 * relative_gap), 3)
    
    return {
        "top_similarity": similarity,
        "confidence": confidence,
        "gap": round(top_score - scores[1], 3) if len(scores) > 1 else None
    }

In [41]:
# def decide_workflow(confidence):
#     if confidence >= 0.75:
#         return "AUTO_EXECUTE_INTENT"
#     elif confidence >= 0.50:
#         return "ASK_CLARIFICATION"
#     else:
#         return "FALLBACK_TO_HUMAN"
    
def decide_workflow(similarity, confidence):
    
    # High similarity + high confidence = clear match
    if similarity >= 0.85 and confidence >= 0.70:
        return "AUTO_EXECUTE_INTENT"
    
    # High similarity + low confidence = ambiguous (multiple good matches)
    elif similarity >= 0.80 and confidence < 0.50:
        return "ASK_CLARIFICATION"  # Multiple good options
    
    # Low similarity = poor match regardless of confidence
    elif similarity < 0.60:
        return "FALLBACK_TO_HUMAN"
    
    # Medium zone - ask for clarification
    else:
        return "ASK_CLARIFICATION"


In [42]:
# final eval 

# New testing with multiple queries:
def print_detailed_results(query, results, metrics, workflow, actual_label=None):
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    if actual_label is not None:
        print(f"Actual Label: {actual_label}")
    print(f"{'='*80}")
    print(f"\nTop Similarity: {metrics['top_similarity']} (how close the match is)")
    print(f"Confidence: {metrics['confidence']} (how certain we are)")
    
    if metrics['gap']:
        print(f"Gap to 2nd: {metrics['gap']} (margin over alternatives)")
    
    print(f"\nWorkflow Decision: {workflow}")
    
    # Explanation logic
    if metrics['top_similarity'] >= 0.85 and metrics['confidence'] >= 0.70:
        print(" High quality match with clear winner - auto-executing")
    elif metrics['top_similarity'] >= 0.80 and metrics['confidence'] < 0.50:
        print("  Good match but multiple similar alternatives - asking for clarification")
    elif metrics['top_similarity'] < 0.60:
        print(" Poor match quality - escalating to human")
    else:
        print("  Uncertain - asking for clarification")
    
    # Check if prediction matches actual label
    predicted_label = results[0].payload['label'] if results else None
    if actual_label is not None and predicted_label is not None:
        if predicted_label == actual_label:
            print(f"\n CORRECT: Predicted label {predicted_label} matches actual label {actual_label}")
        else:
            print(f"\n INCORRECT: Predicted {predicted_label}, but actual label is {actual_label}")
    
    print(f"\nTop 3 Semantic Matches:")
    for i, r in enumerate(results[:3], 1):
        label_match = "✓" if r.payload['label'] == results[0].payload['label'] else "✗"
        print(f"  {i}. [{label_match}] Score: {round(r.score, 3)} | Label: {r.payload['label']} | Text: {r.payload['text']}")


# ==============================================================================
# TEST QUERIES
# ==============================================================================
test_queries = [
    {"query": "I lost my debit card", "actual_label": 41},
    {"query": "I want to change my card pin", "actual_label": 21},
    {"query": "How much cash can I deposit in one go?", "actual_label": 58},
    {"query": "Where is the nearest branch?", "actual_label": 3},
    {"query": "My card payment was declined", "actual_label": None},  # If you don't know the label, use None
    {"query": "I want to know my interest rate", "actual_label": 32},
    {"query": "ATM se paisa nahi nikla", "actual_label": 20},
    {"query": "मैंने अपना कार्ड खो दिया है", "actual_label": 41},
]

# Run all test queries
for test_case in test_queries:
    query = test_case["query"]
    actual_label = test_case["actual_label"]
    
    results = semantic_search(query)
    metrics = calculate_metrics(results)
    workflow = decide_workflow(metrics["top_similarity"], metrics["confidence"])
    
    print_detailed_results(query, results, metrics, workflow, actual_label)

# Optional: Summary statistics
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
correct = 0
total = 0
for test_case in test_queries:
    if test_case["actual_label"] is not None:
        total += 1
        results = semantic_search(test_case["query"])
        predicted_label = results[0].payload['label'] if results else None
        if predicted_label == test_case["actual_label"]:
            correct += 1

if total > 0:
    accuracy = (correct / total) * 100
    print(f"Accuracy: {correct}/{total} ({accuracy:.1f}%)")
else:
    print("No labeled test cases to evaluate")


Query: I lost my debit card
Actual Label: 41

Top Similarity: 0.908 (how close the match is)
Confidence: 0.236 (how certain we are)
Gap to 2nd: 0.005 (margin over alternatives)

Workflow Decision: ASK_CLARIFICATION
  Good match but multiple similar alternatives - asking for clarification

 CORRECT: Predicted label 41 matches actual label 41

Top 3 Semantic Matches:
  1. [✓] Score: 0.908 | Label: 41 | Text: help, lost my card
  2. [✗] Score: 0.902 | Label: 18 | Text: lost my card in atm
  3. [✓] Score: 0.9 | Label: 41 | Text: Help! I lost my card!

Query: I want to change my card pin
Actual Label: 21

Top Similarity: 0.906 (how close the match is)
Confidence: 0.235 (how certain we are)
Gap to 2nd: 0.004 (margin over alternatives)

Workflow Decision: ASK_CLARIFICATION
  Good match but multiple similar alternatives - asking for clarification

 CORRECT: Predicted label 21 matches actual label 21

Top 3 Semantic Matches:
  1. [✓] Score: 0.906 | Label: 21 | Text: Can I change my card PIN?
 